In [ ]:
# Imports
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

# ADDED DIFFERENT PIPELINE FOR SMOTE
from sklearn.pipeline import Pipeline as SklearnPipeline  # sklearn Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline     # imblearn Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    precision_score,
    recall_score
)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# TO ADDRESS IMBALANCED DATASET
from imblearn.over_sampling import SMOTE

# Paths
from pathlib import Path
base_path = Path.cwd()
data_path = base_path / 'Dec2023Flights_PreparedData.csv'

# Load data
df = pd.read_csv(data_path)

# WORKING BY UPLOADING THE FILES - FOR OUR SIDE
# df = pd.read_csv('Merged_Flights_Weather.csv')
# df = pd.read_csv('Dec2023Flights_PreparedData.csv')

# Basic cleaning
df = df.replace(r'^\s*$', np.nan, regex=True)

# Target construction: on-time / delayed / cancelled
df['CANCELLED'] = pd.to_numeric(df['CANCELLED'], errors='coerce').fillna(0).astype(int)
df['DEP_DELAY'] = pd.to_numeric(df['DEP_DELAY'], errors='coerce')
df = df.dropna(subset=['DEP_DELAY'])

def make_status(row):
    if row['CANCELLED'] == 1:
        return 'cancelled'
    elif row['DEP_DELAY'] <= 15:
        return 'on_time'
    else:
        return 'delayed'

df['status'] = df.apply(make_status, axis=1)

# Time features from FlightPlannedDateandTime
df['FlightPlannedDateandTime'] = pd.to_datetime(
    df['FlightPlannedDateandTime'], errors='coerce'
)
df = df.dropna(subset=['FlightPlannedDateandTime'])

df['dep_hour']  = df['FlightPlannedDateandTime'].dt.hour
df['dep_dow']   = df['FlightPlannedDateandTime'].dt.dayofweek
df['dep_month'] = df['FlightPlannedDateandTime'].dt.month
df['dep_date'] = df['FlightPlannedDateandTime'].dt.day

# REMOVE THIS WHEN RUNNING
# max_rows = 40000

# if len(df) > max_rows:
#     df, _ = train_test_split(
#         df,
#         train_size=max_rows,
#         stratify=df['status'],
#         random_state=0
#     )

# Feature selection
numeric_candidates = [
    'dep_hour', 'dep_dow', 'dep_month',
    'Temperature (F)',
    'Dew Point Temp(F)',
    'Rel. Humidity %',
    'Wind Direction (degrees)',
    'Wind Speed (knots)',
    'Precipitation 1 hour',
    'Pressure Altimeter (in.)',
    'Sea Level Pressure (millibar)',
    'Visibility (miles)',
    'Wind Gust (knots)'
]

categorical_candidates = [
    'OP_UNIQUE_CARRIER',
    'ORIGIN',
    'DEST'
]

numeric_features = [c for c in numeric_candidates if c in df.columns]
categorical_features = [c for c in categorical_candidates if c in df.columns]

X = df[numeric_features + categorical_features]
y = df['status']

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

# Numeric preprocessing
numeric_transformer = SklearnPipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ]
)

# Categorical preprocessing
categorical_transformer = SklearnPipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# Column transformer
preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

# Models
models = {
    'LogisticRegression': LogisticRegression(
        max_iter=200,
        multi_class='multinomial',
        solver='saga',
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        random_state=0,
        n_jobs=-1,
        class_weight='balanced'
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=80,
        learning_rate=0.05,
        random_state=0
    ),
}

# Train and evaluate
results = []

for name, model in models.items():
    clf = ImbPipeline(
        steps=[
            ('preprocess', preprocess),
            ('smote', SMOTE(k_neighbors=3, random_state=0)),
            ('model', model)
        ]
    )

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    cm = confusion_matrix(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')

    results.append({
        'model': name,
        'accuracy': acc,
        'f1_macro': f1,
        'precision': precision,
        'recall': recall,
        'confusion_matrix': cm
    })

metrics_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False)
metrics_df

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model,accuracy,f1_macro,precision,recall,confusion_matrix
1,RandomForest,0.712078,0.383668,0.380740,0.407922,"[[1, 5, 7], [36, 2052, 3827], [117, 4821, 19743]]"
2,GradientBoosting,0.664249,0.366223,0.369835,0.442532,"[[3, 3, 7], [184, 2125, 3606], [794, 5683, 182..."
0,LogisticRegression,0.462021,0.314216,0.376149,0.477892,"[[6, 2, 5], [1061, 3107, 1747], [5016, 8636, 1..."


In [ ]:
# Scaling Experiments

scalers = {
    'StandardScaler': StandardScaler(with_mean=False),  # avoid sparse centering issue
    'MinMaxScaler': MinMaxScaler()
}

scaling_results = []

for scaler_name, scaler in scalers.items():

    numeric_scaling = SklearnPipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', scaler)
        ]
    )

    categorical_processing = SklearnPipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]
    )

    preprocess_scaling = ColumnTransformer(
        transformers=[
            ('num', numeric_scaling, numeric_features),
            ('cat', categorical_processing, categorical_features)
        ]
    )

    clf = ImbPipeline(
        steps=[
            ('preprocess', preprocess_scaling),
            ('smote', SMOTE(k_neighbors=3, random_state=0)),
            ('model', LogisticRegression(
                max_iter=200,
                multi_class='multinomial',
                solver='saga'
            ))
        ]
    )

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    scaling_results.append({
        'scaler': scaler_name,
        'accuracy': acc,
        'f1_macro': f1
    })

scaling_df = pd.DataFrame(scaling_results)
scaling_df

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,scaler,accuracy,f1_macro
0,StandardScaler,0.521154,0.334619
1,MinMaxScaler,0.526283,0.336786


In [ ]:
# Imports
import plotly.express as px
from IPython.display import display

viz_df = df.copy()

# Status counts
status_counts = (
    viz_df['status']
    .value_counts()
    .reset_index()
)
status_counts.columns = ['status', 'count']

fig1 = px.bar(
    status_counts,
    x='status',
    y='count',
    title='Flight Status Counts',
    color='status'
)
fig1.update_layout(showlegend=False)
display(fig1)

mean_delay = (
    viz_df.groupby('dep_date')['DEP_DELAY']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

# Average delay by origin (top 15)
origin_delay = (
    viz_df.groupby('ORIGIN')['DEP_DELAY']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .head(15)
)

fig = px.bar(
    origin_delay,
    x='ORIGIN',
    y='DEP_DELAY',
    title='Average Departure Delay by Origin (Top 15)',
    labels={'DEP_DELAY':'Avg Departure Delay (minutes)'},
    color='DEP_DELAY'
)
fig.update_layout(showlegend=False)
fig.show()

fig2 = px.bar(
    mean_delay,
    x='dep_date',
    y='DEP_DELAY',
    title='Average Departure Delay by date',
    labels={'DEP_DELAY':'Avg Delay (minutes)'},
    color='DEP_DELAY',
    template='none'
)

fig2.show()